In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import diags, csr_matrix
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
import shap
from sklearn.pipeline import Pipeline, TransformerMixin
from sklearn.preprocessing import StandardScaler
from pandas.plotting import parallel_coordinates
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [ ]:
rating_raw = pd.read_csv('users.dat', engine='python', names=['itemid'])
rating_raw.index.name = 'userid'
rating_raw.head()

In [ ]:
df_raw = pd.read_csv('raw-data.csv', encoding='latin1')

In [ ]:
df_raw.head()

In [ ]:
df_raw.loc[:, 'doc.id'] = np.arange(len(df_raw))

In [ ]:
df_raw.head()

In [ ]:
tags_raw = pd.read_csv('mult.dat', engine='python', names=['features'])
tags_raw.index.name = 'itemid'
tags_raw.head()

In [ ]:
item_idx = [int(elem) for i in range(len(rating_raw)) for elem in rating_raw.iloc[i]['itemid'].split()]
user_idx = [i for i in range(len(rating_raw)) for j in range(len(rating_raw.iloc[i]['itemid'].split()))]

interactions_matrix = csr_matrix(
    ...,
    shape=...
)

In [ ]:
feature_idx = [int(elem.split(':')[0]) for i in range(len(tags_raw)) for elem in tags_raw.iloc[i]['features'].split()[1:]]
multiplicities = [int(elem.split(':')[1]) for i in range(len(tags_raw)) for elem in tags_raw.iloc[i]['features'].split()[1:]]
item_idx = [i for i in range(len(tags_raw)) for j in range(int(tags_raw.iloc[i]['features'].split()[0]))]

feature_matrix = csr_matrix(
    (multiplicities, (item_idx, feature_idx)),
    shape=(len(tags_raw), 8000)
)

In [ ]:
interactions_matrix.astype(bool).sum(axis=0)

In [ ]:
plt.spy(interactions_matrix, markersize=0.01)
plt.ylabel('User')
plt.xlabel('Item')
plt.show()

In [ ]:
plt.figure(figsize=(10, 10))
plt.spy(feature_matrix, markersize=0.01)
plt.xlabel('Feature')
plt.ylabel('Item')
plt.show()

In [ ]:
plt.semilogy(np.sort(feature_matrix.astype(bool).astype(int).sum(axis=0).A.squeeze()))
plt.xlabel('Feature')
plt.ylabel('Log occurence')
plt.show()

In [ ]:
plt.semilogy(np.sort(interactions_matrix.sum(axis=0).A.squeeze()))
plt.xlabel('Item')
plt.ylabel('Log occurence')
plt.show()

In [ ]:
vocab = pd.read_csv('vocabulary.dat', engine='python', names=['name'])

In [ ]:
vocab.head()

Least occuring features

In [ ]:
vocab.iloc[np.argsort(feature_matrix.sum(axis=0).A.squeeze())[:10]]

Most occuring features

In [ ]:
vocab.iloc[np.argsort(feature_matrix.sum(axis=0).A.squeeze())[-10:][::-1]]

Let's predict the popularity of a newly introduced article - how many users are going to interact with it.

In [ ]:
y = interactions_matrix.sum(axis=0).A.squeeze()
X = df_raw.copy()

In [ ]:
test_size = int(len(y) * 0.1)

X_train_, X_test_, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=2025)

X_train = feature_matrix[X_train_.index, :]
X_test = feature_matrix[X_test_.index, :]

Make sure there are no items without any features.

In [ ]:
(X_train.sum(axis=1) == 0).sum(), (X_test.sum(axis=1) == 0).sum()

Let's also look at the feature coverage

In [ ]:
(X_train.sum(axis=0) == 0).sum(), (X_test.sum(axis=0) == 0).sum()

In [ ]:
def evaluate(model, X, y):
    '''
    Computes RMSE and MAE scores
    '''
    y_pred = ...
    results = pd.DataFrame.from_dict(
        {
            'index':[model.__class__.__name__],
            'columns': ['RMSE', 'MAE'],
            'data': [[..., ...]],
            'index_names':[None],
            'column_names':[None]
            },
        orient='tight')
    return results

In [ ]:
model_zoo = [LinearRegression(), Ridge(), Lasso(), ElasticNet()]

results_no_preproc = pd.DataFrame()
for model in model_zoo:
    model.fit(X_train, y_train)
    results_no_preproc = pd.concat([results_no_preproc, evaluate(model, X_test, y_test)], axis=0)
    

In [ ]:
results_no_preproc.round(2)

In [ ]:
for model in model_zoo:
    print(model.__class__.__name__)
    print(f'w0={model.intercept_:.2f}, ||w||={np.linalg.norm(model.coef_):.2f}')

In [ ]:
explainer = shap.explainers.Linear(model_zoo[0], X_train, feature_names = vocab['name'])
shap_values = explainer(X_train)

shap.plots.beeswarm(shap_values, max_display=10)

In [ ]:
shap.plots.bar(shap_values, max_display=10)

In [ ]:
index = 1000
X_train_.loc[index]

In [ ]:
X_train_.loc[index]['raw.abstract'].split('.') + X_train_.loc[index]['raw.title'].split('.')

In [ ]:
vocab.iloc[X_train[np.where(X_train_['doc.id'] == index)[0][0], :].indices].values.squeeze()

In [ ]:
shap.plots.waterfall(shap_values[np.where(X_train_['doc.id'] == index)[0][0]], max_display=10)

In [ ]:
shap.plots.scatter(shap_values[:, ["selection", "features"]])

In [ ]:
shap.summary_plot(shap_values, features=X_train, feature_names=vocab.values.squeeze(), plot_type='bar', max_display=10)

$$
\text{tf}(t, d) = \frac{f_{t, d}}{\sum_{t'\in d}f_{t', d}}
$$

$$
\text{idf}(t, D) = \log\left( \frac{N}{1 + |\{d\in D: t\in d\}|} \right) + 1
$$

In [ ]:
class TFIDF_Transformer(TransformerMixin):
    '''
    Apply TF-IDF transformation to the document-term matrix
    '''

    def fit(self, X, y=None, **fit_params):
        self.corpus = X.copy()
        idf_scores = self.corpus.astype(bool).sum(axis=0).A.squeeze()
        self.idf = ...
        return self
    
    def transform(self, X, y=None, **fit_params):
        D = diags(1.0 / X.sum(axis=1).A.squeeze())
        tfidf = ...
        return tfidf

In [ ]:
class DenseTransformer(TransformerMixin):
    """
    Convert sparse matrix to dense np array to apply standard scaler with mean.
    """

    def fit(self, X, y=None, **fit_params):
        return self

    def transform(self, X, y=None, **fit_params):
        return X.toarray()

In [ ]:
word_vectorizer = Pipeline([
    ("tfidf", TFIDF_Transformer()), 
    ("dense", DenseTransformer()),
    ("scaler", StandardScaler())
    ])

In [ ]:
word_vectorizer

In [ ]:
model_zoo = [LinearRegression(), Ridge(), Lasso(), ElasticNet()]

feature_matrix_train = word_vectorizer.fit_transform(X_train)
results_tfidf = pd.DataFrame()
for model in model_zoo:
    model.fit(feature_matrix_train, y_train)
    results_tfidf = pd.concat([results_tfidf, evaluate(model, word_vectorizer.transform(X_test), y_test)], axis=0)

results_tfidf.round(2)

In [ ]:
results_no_preproc.round(2)

In [ ]:
model_zoo[0].coef_

In [ ]:
for model in model_zoo:
    print(model.__class__.__name__)
    print(f'w0={model.intercept_:.2f}, ||w||={np.linalg.norm(model.coef_):.2f}')

In [ ]:
plt.figure(figsize=(10, 10))
weights_df = pd.DataFrame.from_dict({model.__class__.__name__: np.argsort(-model.coef_) for model in model_zoo})
weights_df['names'] = vocab.iloc[weights_df['Lasso']]


parallel_coordinates(weights_df,
                     class_column='names',
                     cols=[
                         'ElasticNet',
                         'Lasso',
                         'Ridge',
                         'LinearRegression'
                         ],
                     linewidth=1)
plt.ylabel('Position')
plt.gca().legend_.remove()

# Pop-index

For each user, $\textit{pop-index}$ is the largest value of $p$ such that $p\%$ of items that user has interacted with have also received interactions from $p\%$ of other users.

In [ ]:
item_interactions = (100 * interactions_matrix.sum(axis=0) / (interactions_matrix.shape[0] - 1)).A.squeeze()

user_pop_index = np.zeros(interactions_matrix.shape[0])

indices = interactions_matrix.indices
indptr = interactions_matrix.indptr

for i in range(len(indptr) - 1):
    user_profile = item_interactions[indices[indptr[i]:indptr[i + 1]]]
    user_pop_index[i] = 100 * sum(x >= 100 * (i + 1) / len(user_profile) for i, x in enumerate(sorted(list(user_profile), reverse=True))) / len(user_profile)

In [ ]:
plt.plot(sorted(user_pop_index))
plt.ylabel('Pop-index')
plt.xlabel('User')
plt.show()

In [ ]:
y_bin = np.log(1.0 + (diags(user_pop_index) @ interactions_matrix).max(axis=0).A.squeeze())

y_bin.mean()

In [ ]:
X = feature_matrix.copy()

threshold = 1.5
y_bin = np.log(1.0 + (diags(user_pop_index) @ interactions_matrix).max(axis=0).A.squeeze()) > threshold

X_train, X_test, y_bin_train, y_bin_test = train_test_split(X, y_bin, test_size=test_size, random_state=2025)

In [ ]:
word_vectorizer = Pipeline([
    ("tfidf", TFIDF_Transformer())
    ])

feature_matrix_train = word_vectorizer.fit_transform(X_train)
model = LogisticRegression()
model.fit(feature_matrix_train, y_bin_train)

In [ ]:
np.round(
    accuracy_score(y_true=y_bin_test.astype(int), y_pred=(model.predict_proba(word_vectorizer.transform(X_test)) >= 0.5).astype(int)[:, 1]),
    2)

In [ ]:
np.round(
    accuracy_score(y_true=y_bin_test.astype(int), y_pred=(np.random.rand(len(y_bin_test)) >= 0.5).astype(int)),
    2)